# 6.5 換 tokenizer 後如何比較模型？

# 第 6 章：自己的中英文 tokenizer

前置：字元ID與下一字loss。Tokenizer訓練是統計切詞規則，不是訓練LLM。早期與CLI使用可還原ByteTokenizer；BPE在本章獨立學習，換模型時要同步詞表與checkpoint。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：常見卡片合併：不是先懂詞義**

每次合併一對常見相鄰符號。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/bpe.svg")))

**先想一想：**每token loss較低，但token數多十倍，一定比較好嗎？

換切詞後「每token平均loss」的單位變了。把同一份驗證原文全部負log機率加總，除以原文bytes，才能以同單位對照。

**把直覺寫成數學：**BPB=total negative log likelihood/(UTF8 bytes×ln2)。邊界/BOS/EOS是否計入要保持相同契約。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
import math

raw = "貓🙂"
bytes_count = len(raw.encode())
for name, total_nll in (("示例A", 5.0), ("示例B", 4.0)):
    print(name, total_nll / (bytes_count * math.log(2)), "bits per byte")

**如何讀結果：**示例NLL不是實測模型結果；真正比較使用同一原文、相同文件邊界、各自匹配模型。

**只改一件事：**只把總NLL乘2，驗證BPB倍增。
